# WeightedKgBlend — Path-Gated Re-ranking

**Core idea:** Use RotatE (best KGE model) for candidate generation, then re-rank using
ProbCBR mechanistic path scores specifically computed for RotatE's top-50 candidates.

```
Final score = α × (1/RotatE_rank) + β × ProbCBR_path_score
```

**Inputs (local):**
- `results/predictions/RotatE/slice_X/predictions_{split}.tsv`
- `results/predictions/PathGated/slice_X/path_lookup_rotate_{split}_sliceX.tsv`

**Key result (slice_0):** Path coverage 41.1%, MRR 0.0661 → 0.0754 (+14%), Hits@10 0.1109 → 0.1635 (+47%)

In [ ]:
import pandas as pd
import numpy as np
import optuna
from pathlib import Path
from collections import defaultdict
optuna.logging.set_verbosity(optuna.logging.WARNING)

RESULTS = Path('/Users/meghamala/projects/WeightedKgBlend/results/predictions')
SLICE   = 'slice_0'
TOP_K   = 50

# ── Load RotatE predictions ───────────────────────────────────────────────
rotate_test  = pd.read_csv(RESULTS / 'RotatE' / SLICE / 'predictions_test.tsv',  sep='\t')
rotate_valid = pd.read_csv(RESULTS / 'RotatE' / SLICE / 'predictions_valid.tsv', sep='\t')

# ── Load path scores (ProbCBR scored on RotatE's candidates) ─────────────
i = int(SLICE.split('_')[1])

def load_path_scores(split):
    df = pd.read_csv(RESULTS / 'PathGated' / SLICE / f'path_lookup_rotate_{split}_slice{i}.tsv', sep='\t')
    scores = df.groupby(['drug', 'disease'])['path_score'].sum().to_dict()
    best   = df[df.path_rank == 1].set_index(['drug', 'disease'])['path'].to_dict()
    return scores, best

path_scores_test,  best_path_test  = load_path_scores('test')
path_scores_valid, best_path_valid = load_path_scores('valid')

print(f'RotatE test  : {len(rotate_test):,} rows')
print(f'RotatE valid : {len(rotate_valid):,} rows')
print(f'Path pairs (test)  : {len(path_scores_test):,}')
print(f'Path pairs (valid) : {len(path_scores_valid):,}')

# Coverage
for split, rotate_df, path_scores in [
    ('test',  rotate_test,  path_scores_test),
    ('valid', rotate_valid, path_scores_valid)
]:
    covered, total = 0, 0
    for _, row in rotate_df.iterrows():
        drug = row['drug']
        for k in range(1, 11):
            d = row.get(f'top{k}_disease', '')
            if d:
                total += 1
                if (drug, d) in path_scores: covered += 1
    print(f'Top-10 path coverage ({split}): {covered}/{total} = {covered/total:.1%}')

In [ ]:
# ── Scoring functions ─────────────────────────────────────────────────────

def compute_metrics(rotate_df, path_scores, alpha, beta, top_k=TOP_K):
    rrs, ranks = [], []
    for _, row in rotate_df.iterrows():
        drug, exp_dis = row['drug'], row['expected_disease']
        cands = {}
        for k in range(1, top_k + 1):
            d = row.get(f'top{k}_disease', '')
            if d:
                cands[d] = alpha * (1.0 / k) + beta * path_scores.get((drug, d), 0.0)
        if exp_dis not in cands:
            cands[exp_dis] = alpha * (1.0 / (top_k + 1)) + beta * path_scores.get((drug, exp_dis), 0.0)
        ranked = sorted(cands, key=cands.get, reverse=True)
        rank   = ranked.index(exp_dis) + 1
        rrs.append(1.0 / rank)
        ranks.append(rank)
    ranks = np.array(ranks)
    return {
        'MRR'    : float(np.mean(rrs)),
        'Hits@1' : float((ranks <= 1).mean()),
        'Hits@3' : float((ranks <= 3).mean()),
        'Hits@5' : float((ranks <= 5).mean()),
        'Hits@10': float((ranks <= 10).mean()),
    }


# Baselines
print('Baselines (RotatE within top-50 pool):')
for split, rotate_df in [('valid', rotate_valid), ('test', rotate_test)]:
    m = compute_metrics(rotate_df, {}, 1.0, 0.0)
    print(f'  {split}: MRR={m["MRR"]:.4f}  H@1={m["Hits@1"]:.4f}  '
          f'H@3={m["Hits@3"]:.4f}  H@5={m["Hits@5"]:.4f}  H@10={m["Hits@10"]:.4f}')

In [ ]:
# ── Optuna: optimize alpha and beta on valid set ──────────────────────────

def objective(trial):
    alpha = trial.suggest_float('alpha', 0.0, 1.0)
    beta  = trial.suggest_float('beta',  0.0, 1.0)
    return compute_metrics(rotate_valid, path_scores_valid, alpha, beta)['MRR']

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=300, show_progress_bar=True)

best_alpha = study.best_params['alpha']
best_beta  = study.best_params['beta']

print(f'\nBest alpha (RotatE weight) : {best_alpha:.4f}')
print(f'Best beta  (path weight)   : {best_beta:.4f}')
print(f'Best valid MRR             : {study.best_value:.4f}')

In [ ]:
# ── Final results ─────────────────────────────────────────────────────────

valid_m = compute_metrics(rotate_valid, path_scores_valid, best_alpha, best_beta)
test_m  = compute_metrics(rotate_test,  path_scores_test,  best_alpha, best_beta)
base_v  = compute_metrics(rotate_valid, {}, 1.0, 0.0)
base_t  = compute_metrics(rotate_test,  {}, 1.0, 0.0)

print('=' * 65)
print(f'Results — {SLICE}')
print('=' * 65)
print(f'{"Method":<20} {"Split":<6} {"MRR":>7} {"H@1":>7} {"H@3":>7} {"H@5":>7} {"H@10":>7}')
print('-' * 65)
for split, m in [("valid", base_v), ("test", base_t)]:
    print(f'{"RotatE":<20} {split:<6} {m["MRR"]:>7.4f} {m["Hits@1"]:>7.4f} '
          f'{m["Hits@3"]:>7.4f} {m["Hits@5"]:>7.4f} {m["Hits@10"]:>7.4f}')
print('-' * 65)
for split, m in [("valid", valid_m), ("test", test_m)]:
    print(f'{"Path-Gated":<20} {split:<6} {m["MRR"]:>7.4f} {m["Hits@1"]:>7.4f} '
          f'{m["Hits@3"]:>7.4f} {m["Hits@5"]:>7.4f} {m["Hits@10"]:>7.4f}')
print('=' * 65)
print(f'\nWeights: alpha={best_alpha:.4f} (RotatE), beta={best_beta:.4f} (path)')
print(f'MRR improvement : {(test_m["MRR"]-base_t["MRR"])/base_t["MRR"]*100:+.1f}%')
print(f'H@10 improvement: {(test_m["Hits@10"]-base_t["Hits@10"])/base_t["Hits@10"]*100:+.1f}%')

In [ ]:
# ── Save re-ranked predictions with mechanistic paths ─────────────────────

OUT_DIR = RESULTS / 'PathGated' / SLICE

def build_reranked_df(rotate_df, path_scores, best_path, alpha, beta, top_k=TOP_K):
    rows = []
    for _, row in rotate_df.iterrows():
        drug, exp_dis = row['drug'], row['expected_disease']
        cands = {}
        for k in range(1, top_k + 1):
            d = row.get(f'top{k}_disease', '')
            if d:
                cands[d] = alpha * (1.0 / k) + beta * path_scores.get((drug, d), 0.0)
        if exp_dis not in cands:
            cands[exp_dis] = alpha * (1.0 / (top_k + 1)) + beta * path_scores.get((drug, exp_dis), 0.0)
        ranked = sorted(cands, key=cands.get, reverse=True)
        rank   = ranked.index(exp_dis) + 1
        out = {
            'drug': drug, 'expected_disease': exp_dis,
            'rank': rank, 'reciprocal_rank': 1.0 / rank,
        }
        for k in range(1, top_k + 1):
            d = ranked[k - 1] if k - 1 < len(ranked) else ''
            out[f'top{k}_disease']  = d
            out[f'top{k}_path']     = best_path.get((drug, d), '') if d else ''
            out[f'top{k}_has_path'] = (drug, d) in path_scores if d else False
        rows.append(out)
    return pd.DataFrame(rows)


for split, rotate_df, path_scores, best_path in [
    ('valid', rotate_valid, path_scores_valid, best_path_valid),
    ('test',  rotate_test,  path_scores_test,  best_path_test),
]:
    df = build_reranked_df(rotate_df, path_scores, best_path, best_alpha, best_beta)
    out_path = OUT_DIR / f'predictions_{split}.tsv'
    df.to_csv(out_path, sep='\t', index=False)

    covered = sum(1 for _, row in df.iterrows()
                  for k in range(1, 11) if row.get(f'top{k}_has_path', False))
    total   = sum(1 for _, row in df.iterrows()
                  for k in range(1, 11) if row.get(f'top{k}_disease', ''))
    print(f'{split}: saved {len(df)} rows  |  top-10 path coverage: {covered}/{total} = {covered/total:.1%}')

print('\nDone.')

In [ ]:
# ── Example: show top-5 predictions with mechanistic paths for a drug ─────

df_test = build_reranked_df(rotate_test, path_scores_test, best_path_test, best_alpha, best_beta)

# Find a drug where path-gating improved the rank
rotate_rank_map = rotate_test.set_index(['drug', 'expected_disease'])['rank'].to_dict()
df_test['rotate_rank'] = df_test.apply(
    lambda r: rotate_rank_map.get((r['drug'], r['expected_disease']), 999), axis=1)
df_test['improvement'] = df_test['rotate_rank'] - df_test['rank']

improved = df_test[df_test['improvement'] > 0].sort_values('improvement', ascending=False)
print(f'Drugs where path-gating improved rank: {len(improved)} / {len(df_test)}')
print()

example = improved.iloc[0]
print(f'Drug            : {example["drug"]}')
print(f'Expected disease: {example["expected_disease"]}')
print(f'RotatE rank     : {int(example["rotate_rank"])}')
print(f'Path-Gated rank : {int(example["rank"])}')
print(f'Improvement     : +{int(example["improvement"])} positions')
print()
print('Top-5 after re-ranking:')
print(f'{"#":<4} {"Disease":<35} {"Path"}')
print('-' * 100)
for k in range(1, 6):
    d = example.get(f'top{k}_disease', '')
    p = example.get(f'top{k}_path', '')
    marker = ' ← EXPECTED' if d == example['expected_disease'] else ''
    print(f'{k:<4} {str(d):<35} {str(p)[:55]}{marker}')